# EDA multivariable por estaciones — Bogotá 2020

Este notebook replica la estructura del EDA trabajado previamente, pero queda parametrizado para procesar automáticamente las bases cargadas de **Bogotá** para un año.

**Variables incluidas:**

- PM10
- PM2.5
- RGlobal
- TAire2
- PLiquida
- VViento
- DViento
- HAire2

El flujo realiza:

1. Carga y estandarización de archivos.
2. Limpieza de nombres de estaciones, fechas y valores numéricos.
3. Diagnóstico de duplicados, nulos, negativos y cobertura temporal.
4. Tablas de faltantes por variable y estación.
5. Identificación de huecos temporales.
6. Gráficos exploratorios por variable y estación.
7. Diagnóstico de series de tiempo: ACF, PACF, ADF y STL cuando aplica.
8. Integración multivariable por estaciones comunes.
9. Exportación automática de tablas y bases procesadas.


In [ ]:
# ============================================================
# 0. Librerías
# ============================================================

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

try:
    from statsmodels.tsa.stattools import adfuller
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    from statsmodels.tsa.seasonal import STL
    STATSMODELS_OK = True
except Exception:
    STATSMODELS_OK = False
    print("Advertencia: statsmodels no está instalado. Se omitirán ACF, PACF, ADF y STL.")

## 1. Configuración general

Si ejecutas este notebook en Google Colab, primero sube los CSV al entorno.  
Si lo ejecutas localmente, deja los archivos en la misma carpeta del notebook o cambia las rutas en el diccionario `ARCHIVOS`.

In [ ]:
# ============================================================
# 1. Parámetros del análisis
# ============================================================

CIUDAD = "Bogotá"
ANIO = 2020
FRECUENCIA_ESPERADA = "D"   # D = diario; H = horario si la base viniera por horas
UMBRAL_ROJO = 25            # estaciones con faltantes >= 25%
HUECO_CORTO_MAX_DIAS = 7    # criterio útil para decidir imputación de huecos cortos

# Archivos reales cargados para Bogotá
# Nota: se usa PLiquida como variable de precipitación líquida.
ARCHIVOS = {
    "PM10": "PM10(14).csv",
    "PM25": "PM25(7).csv",
    "RGlobal": "RGlobal(8).csv",
    "TAire2": "TAire2(8).csv",
    "PLiquida": "PLiquida(10).csv",
    "VViento": "VViento(16).csv",
    "DViento": "DViento(13).csv",
    "HAire2": "HAire2(8).csv",
}

# En Colab o local, normalmente estarán en la carpeta actual.
# En este entorno también se revisa /mnt/data.
CARPETAS_BUSQUEDA = [
    Path("."),
    Path("/content"),
    Path("/mnt/data"),
]

CARPETA_SALIDA = Path(f"salidas_EDA_{CIUDAD.replace('ó','o').replace(' ', '_')}_{ANIO}")
CARPETA_SALIDA.mkdir(exist_ok=True)

display(Markdown(f"**Ciudad:** {CIUDAD}  \n**Año:** {ANIO}  \n**Carpeta de salida:** `{CARPETA_SALIDA}`"))


## 2. Funciones auxiliares

Estas funciones permiten que el código funcione aunque existan pequeñas diferencias entre archivos: columnas con comillas, nombres de estaciones con comillas adicionales, columnas `Año`, `Fecha final`, o variables con nombres como `PM2.5`.


In [ ]:
# ============================================================
# 2. Funciones auxiliares
# ============================================================

def resolver_ruta(nombre_archivo, carpetas_busqueda=CARPETAS_BUSQUEDA):
    # Busca un archivo en varias carpetas posibles.
    ruta = Path(nombre_archivo)
    if ruta.exists():
        return ruta

    for carpeta in carpetas_busqueda:
        candidata = carpeta / nombre_archivo
        if candidata.exists():
            return candidata

    raise FileNotFoundError(
        f"No se encontró el archivo {nombre_archivo}. "
        "Verifica que esté subido o cambia la ruta en ARCHIVOS."
    )


def limpiar_estacion(valor):
    # Limpia nombres de estación: quita comillas sobrantes, espacios repetidos y convierte a mayúsculas.
    if pd.isna(valor):
        return np.nan
    valor = str(valor).strip()
    valor = valor.replace('"', "").replace("'", "").strip()
    valor = re.sub(r"\s+", " ", valor)
    return valor.upper()


def detectar_columna_estacion(columnas):
    posibles = [c for c in columnas if c.strip().lower() in ["estacion", "estación"]]
    if posibles:
        return posibles[0]
    raise ValueError("No se encontró columna de estación.")


def detectar_columna_fecha(columnas):
    posibles = [c for c in columnas if "fecha inicial" in c.strip().lower()]
    if posibles:
        return posibles[0]
    posibles = [c for c in columnas if "fecha" in c.strip().lower()]
    if posibles:
        return posibles[0]
    raise ValueError("No se encontró columna de fecha.")


def detectar_columna_valor(df, variable):
    # Detecta la columna de valor de cada archivo.
    columnas = list(df.columns)
    columnas_limpias = {c: c.strip().lower().replace(" ", "_").replace(".", "") for c in columnas}
    objetivo = variable.strip().lower().replace(" ", "_").replace(".", "")

    equivalencias = {
        "PM25": ["pm25", "pm2_5", "pm25"],
    }

    candidatos_objetivo = [objetivo] + equivalencias.get(variable, [])

    for col, col_norm in columnas_limpias.items():
        if col_norm in candidatos_objetivo:
            return col

    columnas_meta = {"año", "anio", "estacion", "estación", "fecha inicial", "fecha final", "fecha"}
    candidatas = [c for c in columnas if c.strip().lower() not in columnas_meta]

    if len(candidatas) == 1:
        return candidatas[0]

    raise ValueError(f"No se pudo detectar columna de valor para {variable}. Columnas disponibles: {columnas}")


def leer_base_variable(nombre_archivo, variable, anio=ANIO):
    # Lee, limpia y estandariza una base de una variable.
    ruta = resolver_ruta(nombre_archivo)
    df = pd.read_csv(ruta, encoding="utf-8-sig")
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

    col_estacion = detectar_columna_estacion(df.columns)
    col_fecha = detectar_columna_fecha(df.columns)
    col_valor = detectar_columna_valor(df, variable)

    base = df[[col_estacion, col_fecha, col_valor]].copy()
    base.columns = ["Estacion", "Fecha", variable]

    base["Estacion"] = base["Estacion"].apply(limpiar_estacion)
    base["Fecha"] = pd.to_datetime(base["Fecha"], errors="coerce")
    base[variable] = pd.to_numeric(base[variable], errors="coerce")
    base["Año"] = base["Fecha"].dt.year

    base = base.dropna(subset=["Estacion", "Fecha"])
    base = base[base["Año"] == anio].copy()
    base = base.sort_values(["Estacion", "Fecha"]).reset_index(drop=True)

    return base


def calendario_anual(anio=ANIO, frecuencia=FRECUENCIA_ESPERADA):
    # Crea calendario esperado del año.
    inicio = f"{anio}-01-01"
    fin = f"{anio}-12-31"
    return pd.date_range(inicio, fin, freq=frecuencia)


def clasificar_faltantes(pct_faltante):
    # Clasificación: Completa, Amarilla o Roja.
    if pd.isna(pct_faltante):
        return "Sin información"
    if pct_faltante == 0:
        return "Completa"
    if pct_faltante < UMBRAL_ROJO:
        return "Amarilla"
    return "Roja"


def estilo_faltantes(tabla):
    # Estilo visual para resaltar porcentaje de faltantes.
    def color_pct(valor):
        try:
            v = float(valor)
        except Exception:
            return ""
        if v >= UMBRAL_ROJO:
            return "background-color: #f8d7da; color: #842029"
        elif v > 0:
            return "background-color: #fff3cd; color: #664d03"
        else:
            return "background-color: #d1e7dd; color: #0f5132"

    columnas = [c for c in tabla.columns if "faltante" in c.lower() or "faltantes" in c.lower()]
    if "pct_faltante" in tabla.columns:
        columnas = ["pct_faltante"]
    return tabla.style.map(color_pct, subset=columnas) if hasattr(tabla.style, "map") else tabla.style.applymap(color_pct, subset=columnas)


def detectar_bloques_huecos(fechas_observadas, fechas_esperadas):
    # Detecta bloques consecutivos de fechas faltantes.
    observadas = set(pd.to_datetime(fechas_observadas).dropna())
    faltantes = sorted([f for f in fechas_esperadas if f not in observadas])

    if not faltantes:
        return []

    bloques = []
    inicio = faltantes[0]
    anterior = faltantes[0]

    for fecha in faltantes[1:]:
        if fecha == anterior + pd.Timedelta(days=1):
            anterior = fecha
        else:
            bloques.append((inicio, anterior, (anterior - inicio).days + 1))
            inicio = fecha
            anterior = fecha

    bloques.append((inicio, anterior, (anterior - inicio).days + 1))
    return bloques


def resumen_diagnostico_variable(df, variable, anio=ANIO):
    # Crea diagnóstico por estación para una variable.
    fechas_esperadas = calendario_anual(anio)
    n_esperado = len(fechas_esperadas)
    filas = []

    for estacion, g in df.groupby("Estacion"):
        fechas_unicas = g["Fecha"].dropna().drop_duplicates()
        registros = len(g)
        registros_unicos_fecha = fechas_unicas.nunique()
        duplicados_fecha = registros - registros_unicos_fecha
        faltantes_calendario = n_esperado - registros_unicos_fecha
        pct_faltante = (faltantes_calendario / n_esperado) * 100

        valores = g[variable]
        filas.append({
            "variable": variable,
            "estacion": estacion,
            "anio": anio,
            "registros": registros,
            "fechas_unicas": registros_unicos_fecha,
            "dias_esperados": n_esperado,
            "faltantes_calendario": faltantes_calendario,
            "pct_faltante": round(pct_faltante, 2),
            "duplicados_fecha": duplicados_fecha,
            "nulos_variable": int(valores.isna().sum()),
            "valores_negativos": int((valores < 0).sum()),
            "ceros": int((valores == 0).sum()),
            "min": valores.min(),
            "max": valores.max(),
            "media": valores.mean(),
            "mediana": valores.median(),
            "desv_std": valores.std(),
            "estado": clasificar_faltantes(round(pct_faltante, 2)),
        })

    return pd.DataFrame(filas).sort_values(["variable", "pct_faltante", "estacion"]).reset_index(drop=True)


def tabla_huecos_variable(df, variable, anio=ANIO):
    # Crea tabla de huecos temporales por estación.
    fechas_esperadas = calendario_anual(anio)
    filas = []

    for estacion, g in df.groupby("Estacion"):
        bloques = detectar_bloques_huecos(g["Fecha"], fechas_esperadas)
        for i, (inicio, fin, duracion) in enumerate(bloques, start=1):
            filas.append({
                "variable": variable,
                "estacion": estacion,
                "bloque": i,
                "inicio_hueco": inicio.date(),
                "fin_hueco": fin.date(),
                "dias_hueco": duracion,
                "tipo_hueco": "Corto/imputable" if duracion <= HUECO_CORTO_MAX_DIAS else "Largo/revisar",
            })

    return pd.DataFrame(filas)


def preparar_serie_diaria(df, variable, estacion):
    # Retorna una serie diaria indexada por fecha. Si hay duplicados, promedia valores por día.
    fechas = calendario_anual(ANIO)
    g = df[df["Estacion"] == estacion].copy()
    serie = g.groupby("Fecha")[variable].mean().reindex(fechas)
    serie.name = f"{variable}_{estacion}"
    return serie

## 3. Carga de bases

El siguiente bloque carga todas las variables configuradas y muestra un resumen inicial de estructura.

In [ ]:
# ============================================================
# 3. Carga de bases
# ============================================================

datos = {}
resumen_archivos = []

for variable, archivo in ARCHIVOS.items():
    base = leer_base_variable(archivo, variable, ANIO)
    datos[variable] = base

    resumen_archivos.append({
        "variable": variable,
        "archivo": archivo,
        "registros": len(base),
        "estaciones": base["Estacion"].nunique(),
        "fecha_min": base["Fecha"].min().date() if len(base) else None,
        "fecha_max": base["Fecha"].max().date() if len(base) else None,
        "anio_detectado": sorted(base["Año"].dropna().unique().tolist()),
        "nulos_variable": int(base[variable].isna().sum()) if len(base) else None,
    })

resumen_archivos = pd.DataFrame(resumen_archivos)
display(resumen_archivos)

resumen_archivos.to_csv(CARPETA_SALIDA / "00_resumen_archivos.csv", index=False, encoding="utf-8-sig")

## 4. Estaciones disponibles por variable

Esta tabla permite verificar qué estaciones tiene cada variable antes de construir una base integrada.

In [ ]:
# ============================================================
# 4. Estaciones disponibles por variable
# ============================================================

estaciones_por_variable = []

for variable, base in datos.items():
    for estacion in sorted(base["Estacion"].dropna().unique()):
        estaciones_por_variable.append({
            "variable": variable,
            "estacion": estacion
        })

estaciones_por_variable = pd.DataFrame(estaciones_por_variable)
matriz_estaciones = (
    estaciones_por_variable
    .assign(disponible=1)
    .pivot_table(index="estacion", columns="variable", values="disponible", fill_value=0)
    .astype(int)
)

display(matriz_estaciones)

matriz_estaciones.to_csv(CARPETA_SALIDA / "01_matriz_estaciones_variables.csv", encoding="utf-8-sig")

## 5. Diagnóstico de cobertura temporal, nulos, duplicados y valores atípicos básicos

La cobertura se calcula contra el calendario completo del año seleccionado.

In [ ]:
# ============================================================
# 5. Diagnóstico por variable y estación
# ============================================================

diagnosticos = []

for variable, base in datos.items():
    diagnosticos.append(resumen_diagnostico_variable(base, variable, ANIO))

diagnostico_general = pd.concat(diagnosticos, ignore_index=True)

display(estilo_faltantes(diagnostico_general))

diagnostico_general.to_csv(CARPETA_SALIDA / "02_diagnostico_general_por_variable_estacion.csv", index=False, encoding="utf-8-sig")

### 5.1 Resumen de estaciones completas, amarillas y rojas

In [ ]:
# ============================================================
# 5.1 Resumen por estado
# ============================================================

resumen_estado = (
    diagnostico_general
    .groupby(["variable", "estado"])
    .size()
    .reset_index(name="cantidad_estaciones")
    .sort_values(["variable", "estado"])
)

display(resumen_estado)

resumen_estado.to_csv(CARPETA_SALIDA / "03_resumen_estado_estaciones.csv", index=False, encoding="utf-8-sig")

## 6. Detección de huecos temporales

Se identifican los días faltantes y se agrupan en bloques consecutivos.  
El criterio `HUECO_CORTO_MAX_DIAS = 7` ayuda a diferenciar huecos pequeños que podrían revisarse para imputación de huecos largos que requieren mayor cuidado.

In [ ]:
# ============================================================
# 6. Huecos temporales
# ============================================================

tablas_huecos = []

for variable, base in datos.items():
    h = tabla_huecos_variable(base, variable, ANIO)
    if not h.empty:
        tablas_huecos.append(h)

if tablas_huecos:
    huecos_general = pd.concat(tablas_huecos, ignore_index=True)
else:
    huecos_general = pd.DataFrame(columns=[
        "variable", "estacion", "bloque", "inicio_hueco", "fin_hueco", "dias_hueco", "tipo_hueco"
    ])

display(huecos_general.head(100))

huecos_general.to_csv(CARPETA_SALIDA / "04_huecos_temporales.csv", index=False, encoding="utf-8-sig")

## 7. Gráficos exploratorios generales

In [ ]:
# ============================================================
# 7.1 Porcentaje de faltantes por estación
# ============================================================

for variable in ARCHIVOS.keys():
    tabla = diagnostico_general[diagnostico_general["variable"] == variable].copy()
    if tabla.empty:
        continue

    tabla = tabla.sort_values("pct_faltante", ascending=True)

    plt.figure(figsize=(12, max(4, 0.35 * len(tabla))))
    plt.barh(tabla["estacion"], tabla["pct_faltante"])
    plt.axvline(UMBRAL_ROJO, linestyle="--", linewidth=1)
    plt.title(f"{CIUDAD} {ANIO} — Porcentaje de faltantes: {variable}")
    plt.xlabel("% faltante respecto al calendario anual")
    plt.ylabel("Estación")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 7.2 Series de tiempo por variable
# ============================================================

for variable, base in datos.items():
    estaciones = sorted(base["Estacion"].dropna().unique())

    plt.figure(figsize=(14, 5))
    for estacion in estaciones:
        serie = preparar_serie_diaria(base, variable, estacion)
        plt.plot(serie.index, serie.values, linewidth=0.9, label=estacion)

    plt.title(f"{CIUDAD} {ANIO} — Serie temporal diaria de {variable}")
    plt.xlabel("Fecha")
    plt.ylabel(variable)
    plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 7.3 Boxplot por estación
# ============================================================

for variable, base in datos.items():
    estaciones = sorted(base["Estacion"].dropna().unique())
    valores = [base.loc[base["Estacion"] == e, variable].dropna().values for e in estaciones]

    if len(estaciones) == 0:
        continue

    plt.figure(figsize=(max(10, 0.7 * len(estaciones)), 5))
    plt.boxplot(valores, labels=estaciones, showfliers=True)
    plt.title(f"{CIUDAD} {ANIO} — Distribución de {variable} por estación")
    plt.ylabel(variable)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 7.4 Promedios mensuales por variable y estación
# ============================================================

resumen_mensual = []

for variable, base in datos.items():
    aux = base.copy()
    aux["mes"] = aux["Fecha"].dt.month

    mensual = (
        aux.groupby(["Estacion", "mes"])[variable]
        .agg(["count", "mean", "median", "min", "max", "std"])
        .reset_index()
    )
    mensual.insert(0, "variable", variable)
    resumen_mensual.append(mensual)

resumen_mensual = pd.concat(resumen_mensual, ignore_index=True)
display(resumen_mensual.head(50))

resumen_mensual.to_csv(CARPETA_SALIDA / "05_resumen_mensual.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ============================================================
# 7.5 Gráficos mensuales
# ============================================================

for variable in ARCHIVOS.keys():
    tabla = resumen_mensual[resumen_mensual["variable"] == variable]

    plt.figure(figsize=(12, 5))
    for estacion, g in tabla.groupby("Estacion"):
        plt.plot(g["mes"], g["mean"], marker="o", linewidth=1, label=estacion)

    plt.title(f"{CIUDAD} {ANIO} — Promedio mensual de {variable}")
    plt.xlabel("Mes")
    plt.ylabel(f"Promedio mensual de {variable}")
    plt.xticks(range(1, 13))
    plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
    plt.tight_layout()
    plt.show()

## 8. Matriz de presencia/ausencia temporal

La matriz permite visualizar rápidamente qué días tienen dato disponible para cada estación.

In [ ]:
# ============================================================
# 8. Matriz de presencia/ausencia por variable
# ============================================================

fechas_esperadas = calendario_anual(ANIO)

for variable, base in datos.items():
    estaciones = sorted(base["Estacion"].dropna().unique())
    matriz = []

    for estacion in estaciones:
        serie = preparar_serie_diaria(base, variable, estacion)
        matriz.append((~serie.isna()).astype(int).values)

    if not matriz:
        continue

    matriz = np.array(matriz)

    plt.figure(figsize=(14, max(3, 0.35 * len(estaciones))))
    plt.imshow(matriz, aspect="auto", interpolation="nearest")
    plt.title(f"{CIUDAD} {ANIO} — Presencia/ausencia de datos: {variable}")
    plt.xlabel("Día del año")
    plt.ylabel("Estación")
    plt.yticks(range(len(estaciones)), estaciones)
    plt.colorbar(label="1 = dato disponible, 0 = dato faltante")
    plt.tight_layout()
    plt.show()

## 9. Análisis detallado por estación

Se grafica cada estación de cada variable con promedio móvil de 7 días.

In [ ]:
# ============================================================
# 9. Serie por estación con media móvil de 7 días
# ============================================================

for variable, base in datos.items():
    estaciones = sorted(base["Estacion"].dropna().unique())

    for estacion in estaciones:
        serie = preparar_serie_diaria(base, variable, estacion)
        media_movil = serie.rolling(window=7, min_periods=1).mean()

        plt.figure(figsize=(13, 4))
        plt.plot(serie.index, serie.values, linewidth=0.9, label="Serie diaria")
        plt.plot(media_movil.index, media_movil.values, linewidth=2, label="Media móvil 7 días")
        plt.title(f"{CIUDAD} {ANIO} — {variable} — {estacion}")
        plt.xlabel("Fecha")
        plt.ylabel(variable)
        plt.legend()
        plt.tight_layout()
        plt.show()

## 10. Diagnóstico de series de tiempo: ADF, ACF, PACF y STL

Este bloque se ejecuta para estaciones con cobertura suficiente.  
Cuando hay pocos datos o demasiados faltantes, se omite el diagnóstico para evitar interpretaciones débiles.

In [ ]:
# ============================================================
# 10. Diagnóstico de series de tiempo
# ============================================================

if not STATSMODELS_OK:
    print("No se ejecuta esta sección porque statsmodels no está disponible.")
else:
    resultados_adf = []

    candidatas = diagnostico_general[
        (diagnostico_general["pct_faltante"] <= UMBRAL_ROJO) &
        (diagnostico_general["fechas_unicas"] >= 60)
    ].copy()

    for _, fila in candidatas.iterrows():
        variable = fila["variable"]
        estacion = fila["estacion"]
        base = datos[variable]
        serie = preparar_serie_diaria(base, variable, estacion)

        # Interpolación temporal solo para diagnóstico gráfico/estadístico.
        # No reemplaza la base original.
        serie_diag = serie.interpolate(method="time", limit_direction="both")
        serie_diag = serie_diag.dropna()

        if len(serie_diag) < 60 or serie_diag.nunique() <= 1:
            continue

        try:
            adf = adfuller(serie_diag)
            resultados_adf.append({
                "variable": variable,
                "estacion": estacion,
                "adf_statistic": adf[0],
                "p_value": adf[1],
                "rezagos": adf[2],
                "n_observaciones": adf[3],
                "interpretacion": "Estacionaria" if adf[1] < 0.05 else "No estacionaria"
            })
        except Exception as e:
            resultados_adf.append({
                "variable": variable,
                "estacion": estacion,
                "adf_statistic": np.nan,
                "p_value": np.nan,
                "rezagos": np.nan,
                "n_observaciones": len(serie_diag),
                "interpretacion": f"No calculado: {e}"
            })

        plt.figure(figsize=(10, 4))
        plot_acf(serie_diag, lags=min(40, len(serie_diag)//2 - 1))
        plt.title(f"ACF — {variable} — {estacion}")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 4))
        plot_pacf(serie_diag, lags=min(40, len(serie_diag)//2 - 1), method="ywm")
        plt.title(f"PACF — {variable} — {estacion}")
        plt.tight_layout()
        plt.show()

        if len(serie_diag) >= 60:
            try:
                stl = STL(serie_diag, period=7, robust=True).fit()
                fig = stl.plot()
                fig.set_size_inches(12, 7)
                fig.suptitle(f"Descomposición STL semanal — {variable} — {estacion}", y=1.02)
                plt.tight_layout()
                plt.show()
            except Exception as e:
                print(f"No fue posible calcular STL para {variable} - {estacion}: {e}")

    resultados_adf = pd.DataFrame(resultados_adf)
    display(resultados_adf)
    resultados_adf.to_csv(CARPETA_SALIDA / "06_resultados_adf.csv", index=False, encoding="utf-8-sig")

## 11. Integración multivariable

Primero se revisan las estaciones comunes entre todas las variables.

Para Bogotá 2020, el notebook construye una **base integrada multivariable** usando las estaciones comunes entre PM10, PM2.5 y las variables meteorológicas seleccionadas: `RGlobal`, `TAire2`, `PLiquida`, `VViento`, `DViento` y `HAire2`.


In [ ]:
# ============================================================
# 11. Integración multivariable
# ============================================================

estaciones_sets = {
    variable: set(base["Estacion"].dropna().unique())
    for variable, base in datos.items()
}

estaciones_comunes_todas = set.intersection(*estaciones_sets.values()) if estaciones_sets else set()
VARIABLES_INTEGRACION = list(ARCHIVOS.keys())

print("Variables incluidas en la integración:")
print(VARIABLES_INTEGRACION)

print("\nEstaciones comunes entre TODAS las variables:")
print(sorted(estaciones_comunes_todas))


def construir_base_integrada(variables, estaciones_comunes, nombre_salida):
    # Construye una base integrada por Fecha + Estacion.
    fechas = calendario_anual(ANIO)

    base_integrada = []
    for estacion in sorted(estaciones_comunes):
        tmp = pd.DataFrame({
            "Fecha": fechas,
            "Estacion": estacion
        })
        base_integrada.append(tmp)

    if not base_integrada:
        return pd.DataFrame()

    base_integrada = pd.concat(base_integrada, ignore_index=True)

    for variable in variables:
        base = datos[variable].copy()
        aux = (
            base[base["Estacion"].isin(estaciones_comunes)]
            .groupby(["Fecha", "Estacion"], as_index=False)[variable]
            .mean()
        )
        base_integrada = base_integrada.merge(aux, on=["Fecha", "Estacion"], how="left")

    base_integrada["Año"] = base_integrada["Fecha"].dt.year
    base_integrada["Mes"] = base_integrada["Fecha"].dt.month
    base_integrada["Dia"] = base_integrada["Fecha"].dt.day
    base_integrada["DiaAnio"] = base_integrada["Fecha"].dt.dayofyear

    ruta_salida = CARPETA_SALIDA / nombre_salida
    base_integrada.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    return base_integrada


base_integrada = construir_base_integrada(
    VARIABLES_INTEGRACION,
    estaciones_comunes_todas,
    "07_base_integrada_multivariable.csv"
)

# Se mantiene este alias para reutilizar las celdas siguientes del notebook.
base_integrada_principal = base_integrada
estaciones_comunes_principal = estaciones_comunes_todas
VARIABLES_INTEGRACION_PRINCIPAL = VARIABLES_INTEGRACION

print("\nDimensión base integrada multivariable:", base_integrada.shape)

display(base_integrada.head())


### 11.1 Diagnóstico de la base integrada multivariable


In [ ]:
# ============================================================
# 11.1 Diagnóstico de base integrada multivariable
# ============================================================

if not base_integrada.empty:
    diagnostico_integrada = []

    for estacion, g in base_integrada.groupby("Estacion"):
        fila = {
            "estacion": estacion,
            "registros": len(g),
            "fecha_min": g["Fecha"].min().date(),
            "fecha_max": g["Fecha"].max().date(),
        }
        for variable in VARIABLES_INTEGRACION:
            fila[f"nulos_{variable}"] = int(g[variable].isna().sum())
            fila[f"pct_nulos_{variable}"] = round(g[variable].isna().mean() * 100, 2)
        diagnostico_integrada.append(fila)

    diagnostico_integrada = pd.DataFrame(diagnostico_integrada)
    display(diagnostico_integrada)

    diagnostico_integrada.to_csv(
        CARPETA_SALIDA / "08_diagnostico_base_integrada_multivariable.csv",
        index=False,
        encoding="utf-8-sig"
    )
else:
    print("No se construyó la base integrada porque no hay estaciones comunes entre todas las variables.")


## 12. Análisis de la base integrada multivariable

Estos gráficos permiten revisar la relación temporal entre contaminantes y variables meteorológicas en las estaciones comunes.

In [ ]:
# ============================================================
# 12. Gráficos de la base integrada multivariable
# ============================================================

if not base_integrada_principal.empty:
    for estacion, g in base_integrada_principal.groupby("Estacion"):
        g = g.sort_values("Fecha")

        for variable in VARIABLES_INTEGRACION_PRINCIPAL:
            plt.figure(figsize=(13, 4))
            plt.plot(g["Fecha"], g[variable], linewidth=0.9)
            plt.title(f"Base integrada — {CIUDAD} {ANIO} — {estacion} — {variable}")
            plt.xlabel("Fecha")
            plt.ylabel(variable)
            plt.tight_layout()
            plt.show()

In [ ]:
# ============================================================
# 12.1 Correlaciones en base integrada multivariable
# ============================================================

if not base_integrada_principal.empty:
    for estacion, g in base_integrada_principal.groupby("Estacion"):
        corr = g[VARIABLES_INTEGRACION_PRINCIPAL].corr(numeric_only=True)
        display(Markdown(f"### Matriz de correlación — {estacion}"))
        display(corr)

        plt.figure(figsize=(8, 6))
        plt.imshow(corr, interpolation="nearest")
        plt.title(f"Matriz de correlación — {estacion}")
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.index)), corr.index)
        plt.colorbar(label="Correlación")

        for i in range(len(corr.index)):
            for j in range(len(corr.columns)):
                valor = corr.iloc[i, j]
                if pd.notna(valor):
                    plt.text(j, i, f"{valor:.2f}", ha="center", va="center")

        plt.tight_layout()
        plt.show()

## 13. Conclusiones automáticas del EDA

Este bloque genera una síntesis inicial para documentar el avance del objetivo específico relacionado con la construcción de la base de datos integrada.

In [ ]:
# ============================================================
# 13. Conclusiones automáticas
# ============================================================

total_variables = len(ARCHIVOS)
total_estaciones = len(set().union(*estaciones_sets.values()))

completas = diagnostico_general[diagnostico_general["estado"] == "Completa"].shape[0]
amarillas = diagnostico_general[diagnostico_general["estado"] == "Amarilla"].shape[0]
rojas = diagnostico_general[diagnostico_general["estado"] == "Roja"].shape[0]

display(Markdown(f'''
## Síntesis del EDA — {CIUDAD} {ANIO}

- Se analizaron **{total_variables} variables**: {", ".join(ARCHIVOS.keys())}.
- Se identificaron **{total_estaciones} estaciones únicas** en el conjunto de archivos.
- En el diagnóstico por variable-estación se encontraron:
  - **{completas}** casos completos.
  - **{amarillas}** casos con faltantes menores al {UMBRAL_ROJO}%.
  - **{rojas}** casos con faltantes iguales o superiores al {UMBRAL_ROJO}%.
- Las estaciones comunes entre todas las variables fueron: **{sorted(estaciones_comunes_todas) if estaciones_comunes_todas else "ninguna"}**.
- La base integrada multivariable se construyó con: **{", ".join(VARIABLES_INTEGRACION)}**.
- La base integrada contiene **{base_integrada.shape[0] if not base_integrada.empty else 0} registros** y **{base_integrada.shape[1] if not base_integrada.empty else 0} columnas**.
- Las tablas, diagnósticos y bases procesadas fueron exportadas en la carpeta: `{CARPETA_SALIDA}`.
'''))


## 14. Archivos exportados

Al finalizar la ejecución, se generan archivos CSV en la carpeta de salida:

- `00_resumen_archivos.csv`
- `01_matriz_estaciones_variables.csv`
- `02_diagnostico_general_por_variable_estacion.csv`
- `03_resumen_estado_estaciones.csv`
- `04_huecos_temporales.csv`
- `05_resumen_mensual.csv`
- `06_resultados_adf.csv` si `statsmodels` está disponible
- `07_base_integrada_multivariable.csv`
- `08_diagnostico_base_integrada_multivariable.csv`
